In [1]:
#@title { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r'''
<div style="padding:28px 30px; border:1px solid #d6e6ff; border-radius:24px; background:linear-gradient(180deg,#f8fbff 0%,#f1f7ff 100%);">
  <div style="display:flex;align-items:flex-start;justify-content:space-between;gap:20px;margin-bottom:10px;">
  <div style="display:inline-block; margin-bottom:0; padding:6px 12px; border-radius:999px; background:rgba(36,94,168,0.10); border:1px solid rgba(36,94,168,0.16); color:#245ea8; font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase;">AI культура · трек 2 · занятие 4</div>
    <a href="https://nexign.com/ru" target="_blank" rel="noopener" title="Nexign" style="display:block;flex:0 0 auto;text-decoration:none;line-height:1;">
      <img src="https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/AI%20Culture/QualifiedCustomer/lesson04/Logo_Nexign_Green.svg" alt="Nexign" style="display:block;width:108px;height:auto;border:0;">
    </a>
  </div>
  <h1 style="margin:0 0 12px; font-size:30px; line-height:1.18; color:#16385f;"><strong>Занятие 4. Практика в блокноте</strong></h1>
  <p style="margin:0; font-size:17px; color:#4f6f96; max-width:920px;">Переводим очищенную таблицу в machine-ready формат: определяем роли столбцов, кодируем категории, выбираем масштабирование и собираем итоговый CSV.</p>
</div>
'''))

In [2]:
#@title { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r'''
<div style="margin:18px 0; padding:18px 20px; border:1px solid #d6e6ff; border-radius:18px; background:#ffffff;">
  <div style="font-size:12px; font-weight:700; text-transform:uppercase; letter-spacing:0.04em; color:#245ea8; margin-bottom:8px;">Как работать с блокнотом</div>
  <div style="font-size:15px; line-height:1.55; color:#2b2b2b;">Идите по шагам: аудит → роли столбцов → типы → кодирование → масштабирование → итоговый CSV. Для домашней работы повторите эту логику на своём датасете из занятий 2–3.</div>
</div>
'''))

## План работы

1. Загрузить очищенный, но ещё не machine-ready набор.
2. Определить роли столбцов и проверить типы.
3. Закодировать категориальные признаки.
4. Сравнить варианты масштабирования числовых признаков.
5. Собрать и сохранить итоговый `machine-ready CSV`.

## Цель практики

Подготовить таблицу к дальнейшему машинному анализу:

```text
аудит → типы → кодирование → масштабирование → machine-ready CSV
```

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)
plt.rcParams['figure.figsize'] = (10, 4)

BASE_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/AI%20Culture/QualifiedCustomer/lesson04"

def read_lesson_csv(filename):
    local_path = Path(filename)
    source = local_path if local_path.exists() else f"{BASE_URL}/{filename}"
    return pd.read_csv(source)


## 4. Основная практика: от очищенной таблицы к machine-ready CSV

Теперь работаем с `lesson04_salary_transform_dirty.csv`.

Это учебный датасет о вакансиях и зарплатных характеристиках, подготовленный специально для демонстрации типизации признаков, кодирования и масштабирования.  
Каждая строка соответствует отдельной записи о позиции, а столбцы описывают регион, профессиональное направление, формат работы, грейд, требование английского языка и несколько числовых характеристик.

Файл не является полностью сырым, но и не представляет собой финальный набор признаков.  
Он специально оставлен в промежуточном состоянии, чтобы на нём можно было последовательно показать:
- как определить роль каждого столбца;
- как исправить и зафиксировать типы;
- как закодировать категориальные признаки;
- как выбрать масштабирование и получить итоговый machine-ready CSV.


In [ ]:
df = read_lesson_csv('lesson04_salary_transform_dirty.csv')
display(df.head())
print('Размер таблицы:', df.shape)
print('\nТипы данных:')
display(df.dtypes.to_frame('dtype'))


In [ ]:
audit = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'non_null': [int(df[c].notna().sum()) for c in df.columns],
    'nunique': [int(df[c].nunique()) for c in df.columns],
    'example': [str(df[c].iloc[0]) for c in df.columns]
})
display(audit)


### 4.1. Сначала смысл столбца, потом его технический тип

На практике полезно различать четыре группы:

- **service** - идентификаторы и служебные поля;
- **categorical** - конечные наборы категорий;
- **binary** - булевы признаки;
- **numeric** - количественные признаки, для которых осмысленны арифметические операции.

Типизация - это содержательное решение.  
Например, числовой код не всегда является количественным признаком, а строковый `yes/no` или `да/нет` часто нужно привести к бинарному виду `0/1`.


In [ ]:
service_cols = ['vacancy_id']
categorical_cols = ['region', 'role_family', 'work_mode', 'grade']
binary_cols = ['english_required']
numeric_cols = ['salary_rub', 'bonus_rub', 'experience_years', 'applicants_count', 'commute_minutes']

type_map = pd.DataFrame({
    'column': service_cols + categorical_cols + binary_cols + numeric_cols,
    'role': (
        ['service'] * len(service_cols) +
        ['categorical'] * len(categorical_cols) +
        ['binary'] * len(binary_cols) +
        ['numeric'] * len(numeric_cols)
    )
})
display(type_map)


In [ ]:
df_prepared = df.copy()
df_prepared['english_required'] = (
    df_prepared['english_required']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({'yes': 1, 'no': 0, 'да': 1, 'нет': 0})
    .astype('Int64')
)
display(df_prepared.head())


### 4.2. Кодирование категорий

Если категориальный признак принимает $k$ значений, one-hot encoding заменяет его набором бинарных индикаторов:
$$
c \in \{c_1,\dots,c_k\}
\quad\longrightarrow\quad
(I[c=c_1],\dots,I[c=c_k])
$$

Это снимает ложный порядок между категориями и превращает категориальный столбец в набор машинно-читаемых признаков.

One-hot имеет смысл использовать там, где число категорий умеренно.  
Идентификаторы, уникальные коды и длинные свободные тексты под такую схему обычно не подходят.


In [ ]:
encoded = pd.get_dummies(
    df_prepared[categorical_cols],
    prefix=categorical_cols,
    drop_first=False
)

display(encoded.head())
print('Количество новых dummy-столбцов:', encoded.shape[1])


### 4.3. Масштабирование числовых признаков

Три базовые формулы:

**Стандартизация**
$$
z = \frac{x-\mu}{\sigma}
$$

**Min-Max**
$$
x' = \frac{x-\min(x)}{\max(x)-\min(x)}
$$

**Robust scaling**
$$
x^{(rob)} = \frac{x-\mathrm{Median}(x)}{IQR}
$$

Выбор зависит от структуры данных:

- `StandardScaler` удобен, когда важна нормированная шкала отклонений;
- `MinMaxScaler` полезен при фиксированном диапазоне значений;
- `RobustScaler` особенно уместен при хвостах и нетипичных значениях.


In [ ]:
scalers = {
    'standard': StandardScaler(),
    'minmax': MinMaxScaler(),
    'robust': RobustScaler()
}

scaled_versions = {}
for name, scaler in scalers.items():
    scaled_arr = scaler.fit_transform(df_prepared[numeric_cols])
    scaled_versions[name] = pd.DataFrame(scaled_arr, columns=[f'{c}__{name}' for c in numeric_cols])

scaler_preview = pd.concat([scaled_versions['standard'].head(), scaled_versions['minmax'].head(), scaled_versions['robust'].head()], axis=1)
display(scaler_preview)


### 4.4. Сборка итогового набора

Итоговый `machine_ready.csv` должен содержать только те столбцы, которые имеют смысл как признаки.
В демонстрационном решении:

- `vacancy_id` исключается из матрицы признаков;
- `english_required` сохраняется как бинарный признак;
- категориальные поля переводятся в one-hot;
- числовые признаки масштабируются через `RobustScaler`.


In [ ]:
final_scaled_numeric = scaled_versions['robust'].copy()
final_scaled_numeric.columns = [c.replace('__robust', '') for c in final_scaled_numeric.columns]

machine_ready = pd.concat(
    [
        df_prepared[['english_required']].astype(int),
        encoded.astype(int),
        final_scaled_numeric
    ],
    axis=1
)

display(machine_ready.head())
print('Размер итоговой матрицы признаков:', machine_ready.shape)


In [ ]:
machine_ready.to_csv('lesson04_machine_ready_demo.csv', index=False)
print('Файл сохранён: lesson04_machine_ready_demo.csv')
print('Первые столбцы:', list(machine_ready.columns[:12]))


## 5. Как проверить, что файл действительно machine-ready

Минимальный набор проверок:

1. у каждого признака корректный тип;
2. категориальные поля закодированы или осмысленно исключены;
3. служебные столбцы не попали в признаки случайно;
4. числовые признаки приведены к выбранной шкале;
5. итоговый CSV можно заново открыть без ручной правки.


## Домашняя работа

Используйте собственный датасет, выбранный на занятии 2 и очищенный на занятии 3. Подготовьте итоговый `machine-ready CSV` и кратко объясните принятые решения по типам, кодированию, логарифмированию и масштабированию.

Полные требования и формат сдачи — в задании Moodle.

In [ ]:
#@title Подготовка дополнительных учебных файлов { display-mode: "form" }
from pathlib import Path
from urllib.parse import quote
from urllib.request import urlretrieve

_EXTRA_DATA_BASE = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/AI%20Culture/QualifiedCustomer/lesson04"
_EXTRA_DATA_FILES = ["CustomerPurchaseDataset.csv", "Loan_approval_data_2025.csv"]
for _filename in _EXTRA_DATA_FILES:
    _path = Path(_filename)
    if not _path.exists():
        urlretrieve(f"{_EXTRA_DATA_BASE}/{quote(_filename)}", _filename)
        print(f"Загружен {_filename}")

# Практическое задание: Преобразование категорий и нормализация признаков

Вы работаете аналитиком в банке. Необходимо подготовить данные о заявителях по кредитам для построения модели машинного обучения, которая ***будет прогнозировать решение по кредитной заявке.***

В качестве исходных данных используется датасет `Loan_approval_data_2025.csv`.

**Целевая переменная:**

`loan_status` — результат рассмотрения заявки (например, одобрена или отклонена).

Перед обучением модели необходимо выполнить анализ структуры данных, преобразовать категориальные признаки в числовой формат и подготовить числовые признаки с помощью различных методов масштабирования.

Загрузка датасета:

In [ ]:
# Напишите ваш код здесь

# Задачи для выполнения
# Задача 1: Исследовательский анализ данных
Условие: Проведите исследовательский анализ данных.

Определите:
1. Количество строк и столбцов.
2. Какие признаки являются числовыми, а какие — категориальными.
3. Наличие пропущенных значений.

4. Количество уникальных значений для каждого категориального признака.

5. (*) Построить гистограммы распределения для всех числовых признаков.

6. Сделать краткий вывод о структуре набора данных.

In [ ]:
# Напишите ваш код здесь

# Задача 2: Преобразование категориальных признаков
Перед обучением модели необходимо преобразовать категориальные признаки в числовую форму.

**Требования**
1. Выберите один категориальный признак, имеющий небольшое количество уникальных значений, и выполните для него **Label Encoding**.
2. Выберите один номинальный категориальный признак (например, семейное положение, образование, тип занятости или аналогичный признак из датасета) и выполните **One-Hot Encoding**.
3. Выберите категориальный признак с несколькими повторяющимися категориями и выполните **Frequency Encoding**.
4. Создайте новый DataFrame, содержащий исходные числовые признаки и преобразованные категориальные признаки.
5. Сделайте вывод о том, в каких случаях каждый метод кодирования является наиболее подходящим.

**Примечание.**
` Конкретные признаки выбираются студентом исходя из структуры датасета.`

In [ ]:
# Напишите ваш код здесь

# Задача 3: Сравнение методов кодирования
Сравните полученные способы кодирования.
**Требования:**

1. Определите, насколько увеличилось количество признаков после применения **One-Hot Encoding**.
2. Для каждого метода постройте визуализацию (*).

**Например:**
* **для Label Encoding** — столбчатую диаграмму распределения полученных кодов;
* **для One-Hot Encoding** — тепловую карту корреляций между созданными бинарными признаками;
*  для **Frequency Encoding** — график распределения полученных значений.
3. Сравните преимущества и недостатки каждого способа кодирования.
4. Сделайте вывод, какой метод предпочтительнее использовать для различных типов категориальных признаков.


In [ ]:
# Напишите ваш код здесь

# Задача 4: Масштабирование числовых признаков
Подготовьте числовые признаки к использованию в модели машинного обучения

**Требования**
1. Выберите два числовых признака, характеризующих клиента (например, возраст, доход, стаж работы или аналогичные признаки из датасета), и примените **Min-Max Scaling**.
2. Выберите два других числовых признака и выполните **Standard Scaling**.
3. Для одного числового признака выполните **Robust Scaling** и сравните результат со **Standard Scaling**.
4. (*)Постройте boxplot для выбранных признаков:
* до масштабирования;
* после масштабирования.
5. Рассчитайте основные статистики (**среднее значение, стандартное отклонение, минимум и максимум)** до и после преобразований.
6. Сделайте вывод о том, как различные методы масштабирования влияют на данные и в каких случаях каждый из них рекомендуется использовать.

In [ ]:
# Напишите ваш код здесь

## Итоговое задание

Подготовьте набор данных к дальнейшему использованию в задачах машинного обучения.

В результате работы необходимо:

* получить набор данных, содержащий только признаки, пригодные для обучения модели
* преобразовать выбранные категориальные признаки в числовой формат
* выполнить масштабирование выбранных числовых признаков
* сохранить подготовленный DataFrame для дальнейшего использования

In [ ]:
# Напишите ваш код здесь